# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/vascular`. Старые метрики не смешиваются с актуальными.


In [ ]:
from pathlib import Path
from datetime import datetime, date
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from openpyxl.utils.datetime import to_excel
from scipy.stats import mannwhitneyu, fisher_exact, beta
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

SOURCE_PATH = Path("data/База_док_13_08_2026_Коррекция_по_нозологии_финал_.csv")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "hospital_death"
GLUCOSE_ELEVATED_THRESHOLD_MMOL_L = 5.6

## 1. Загрузка исходной таблицы


In [ ]:
df = pd.read_csv(SOURCE_PATH, sep=";")
print(df.shape[0], df.shape[1])
display(pd.DataFrame({"Первые столбцы": df.columns[:15]}))

In [ ]:
miss     = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(1)
audit    = pd.DataFrame({'missing_n': miss, 'missing_%': miss_pct, 'dtype': df.dtypes})
audit    = audit[audit['missing_n'] > 0].sort_values('missing_%', ascending=False)
print(f'Столбцов с пропусками: {len(audit)}')
display(audit)

empty_cols_raw = [c for c in df.columns if df[c].isna().all()]
print(f'\nПолностью пустых столбцов: {len(empty_cols_raw)}')

space_cols = [c for c in df.columns if c != c.strip()]
print(f'Столбцов с пробелами в имени: {len(space_cols)}')
for c in space_cols:
    print(f'  {repr(c)}')

In [ ]:
idxs = df[df['ФИО'].isna()].index
print(idxs)

In [ ]:
# for i in idxs:
#     print(df.loc[i])

In [ ]:
df.iloc[-3:]

In [ ]:
df_raw = df.drop(index=list(idxs) + [0])
df_raw = df_raw.iloc[:-3]
df_raw = df_raw.reset_index(drop=True)

In [ ]:
df_raw.iloc[-3:]

In [ ]:
print(df_raw['Исход 1- жив,0 -мерт'].unique())
df_raw[df_raw['Исход 1- жив,0 -мерт'].isna()]

In [ ]:
alive_raw = pd.to_numeric(
    df_raw["Исход 1- жив,0 -мерт"],
    errors="coerce"
)

df_raw = df_raw.loc[alive_raw.isin([0, 1])].reset_index(drop=True)

print(len(df_raw))
print(int((pd.to_numeric(df_raw['Исход 1- жив,0 -мерт']) == 0).sum()))

## 2. Очистка

In [ ]:
def parse_operation_dates(series):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.replace({"-": pd.NA, "": pd.NA})
    cleaned = cleaned.str.replace(
        r"^0?2(\d{2})(-\d{2}-\d{2})",
        r"20\1\2",
        regex=True
    )

    return pd.to_datetime(
        cleaned,
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )


def parse_birth_dates(series):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.replace({"-": pd.NA, "": pd.NA})

    birth_date = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    slash_format = cleaned.str.contains("/", na=False)
    dot_format = cleaned.str.contains(r"\.", regex=True, na=False)
    other_format = ~(slash_format | dot_format) & cleaned.notna()

    birth_date.loc[slash_format] = pd.to_datetime(
        cleaned.loc[slash_format],
        errors="coerce",
        dayfirst=False,
        format="mixed"
    )

    birth_date.loc[dot_format] = pd.to_datetime(
        cleaned.loc[dot_format],
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )

    birth_date.loc[other_format] = pd.to_datetime(
        cleaned.loc[other_format],
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )

    return birth_date


def numeric_value(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return float(to_excel(value))
    text = str(value).strip().replace("\u00a0", " ").replace(",", ".")
    match = re.search(r"[-+]?\d+(?:\.\d+)?", text)
    return float(match.group()) if match else np.nan


def numeric_series(series):
    return series.map(numeric_value).astype(float)


def coded_binary(series, blank_means_zero=False):
    normalized = series.astype("string").str.strip().str.lower()
    mapping = {
        "+": 1,
        "1": 1,
        "1.0": 1,
        "да": 1,
        "yes": 1,
        "true": 1,
        "-": 0,
        "0": 0,
        "0.0": 0,
        "нет": 0,
        "no": 0,
        "false": 0,
    }
    result = normalized.map(mapping)
    if blank_means_zero:
        result = result.fillna(0)
    return result.astype("Int64")


def combine_binary_strict(*series_list):
    values = pd.concat(series_list, axis=1)
    result = pd.Series(
        pd.NA,
        index=values.index,
        dtype="Int64"
    )
    any_positive = values.eq(1).any(axis=1)
    all_negative = values.notna().all(axis=1) & values.eq(0).all(axis=1)
    result.loc[any_positive] = 1
    result.loc[all_negative] = 0
    return result


def infer_sex(value):
    if pd.isna(value):
        return pd.NA
    tokens = re.sub(r"[^а-яё -]", " ", str(value).lower()).split()
    if any(token.endswith(("вна", "ична", "инична")) for token in tokens):
        return "female"
    if any(token.endswith(("вич", "ьич")) for token in tokens):
        return "male"
    if tokens and tokens[0].endswith(("ова", "ева", "ёва", "ина", "ына", "ая", "ская", "цкая")):
        return "female"
    if tokens and tokens[0].endswith(("ов", "ев", "ёв", "ин", "ын", "ий", "ый", "ской", "цкий")):
        return "male"
    return pd.NA


def documented_calcification(series):
    normalized = series.astype("string").str.strip().str.lower()
    result = pd.Series(
        pd.NA,
        index=series.index,
        dtype="Int64"
    )
    marker_positive = normalized.isin(["+", "1", "1.0", "да", "yes", "true"])
    marker_negative = normalized.isin(["-", "0", "0.0", "нет", "no", "false"])
    negative_text = normalized.str.contains(
        r"без[^.;]*кальцин|кальцин\w*[^.;]*не выяв",
        regex=True,
        na=False
    )
    positive_text = normalized.str.contains(
        r"кальцин|обызвеств|кальцинир",
        regex=True,
        na=False
    ) & ~negative_text
    result.loc[marker_negative | negative_text] = 0
    result.loc[marker_positive | positive_text] = 1
    return result


operation_date = parse_operation_dates(df_raw["Дата операции"])
birth_date = parse_birth_dates(df_raw["Год рождения"])

future_birth_date = (
    operation_date.notna()
    & birth_date.notna()
    & (birth_date > operation_date)
)

birth_date.loc[future_birth_date] = (
    birth_date.loc[future_birth_date]
    - pd.DateOffset(years=100)
)

valid_dates = operation_date.notna() & birth_date.notna()

birthday_not_reached = (
    (operation_date.dt.month < birth_date.dt.month)
    | (
        (operation_date.dt.month == birth_date.dt.month)
        & (operation_date.dt.day < birth_date.dt.day)
    )
)

df_raw["age_years"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="Int64"
)

df_raw.loc[valid_dates, "age_years"] = (
    operation_date.loc[valid_dates].dt.year
    - birth_date.loc[valid_dates].dt.year
    - birthday_not_reached.loc[valid_dates].astype(int)
).astype("Int64")

df_raw["age_years"] = df_raw["age_years"].where(
    df_raw["age_years"].between(18, 105)
)

df_raw = (
    df_raw
    .dropna(subset=["age_years"])
    .reset_index(drop=True)
)

df_raw["sex"] = df_raw["ФИО"].map(infer_sex).astype("string")
manual_male = (
    df_raw["ФИО"]
    .astype("string")
    .str.contains(
        "ANONYMIZED_PERSON",
        case=False,
        na=False
    )
)
df_raw.loc[manual_male, "sex"] = "male"
df_raw["female"] = (
    df_raw["sex"]
    .map({"female": 1, "male": 0})
    .astype("Int64")
)

df_raw["ef_percent"] = numeric_series(df_raw["Узи сердца ФВ %"])
df_raw["ef_percent"] = df_raw["ef_percent"].where(
    df_raw["ef_percent"] > 1.5,
    df_raw["ef_percent"] * 100
)
df_raw["ef_percent"] = df_raw["ef_percent"].where(
    df_raw["ef_percent"].between(10, 90)
)

df_raw["ef_category"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="string"
)
df_raw.loc[df_raw["ef_percent"] < 40, "ef_category"] = "<40"
df_raw.loc[
    df_raw["ef_percent"].between(40, 50, inclusive="both"),
    "ef_category"
] = "40–50"
df_raw.loc[df_raw["ef_percent"] > 50, "ef_category"] = ">50"

df_raw["stroke_history"] = coded_binary(
    df_raw["ОНМК анамнез (да/нет)"]
)
df_raw["diabetes"] = coded_binary(
    df_raw["диабет"]
)
df_raw["mi_acute_at_index"] = coded_binary(
    df_raw["Инфаркт миокарда острый"]
)
df_raw["mi_history"] = coded_binary(
    df_raw["Инфаркт анамнез (да/нет)"]
)
df_raw["ihd_recorded"] = coded_binary(
    df_raw["ИБС"]
)
df_raw["pci_history"] = coded_binary(
    df_raw["ЧТКА"]
)

af_paroxysmal = coded_binary(
    df_raw["ФП пароксизмальная"]
)
af_other = coded_binary(
    df_raw["ФП"]
)

df_raw["atrial_fibrillation"] = combine_binary_strict(
    af_paroxysmal,
    af_other
)

df_raw["prior_vascular_surgery"] = coded_binary(
    df_raw["любая операция на сосудах до индексного события"]
)

df_raw["ihd_before_index"] = combine_binary_strict(
    df_raw["ihd_recorded"],
    df_raw["mi_history"],
    df_raw["pci_history"]
)

df_raw["mi_any"] = pd.concat(
    [
        df_raw["mi_history"],
        df_raw["mi_acute_at_index"],
    ],
    axis=1
).max(axis=1).astype("Int64")

df_raw["egfr_value"] = numeric_series(df_raw["СКФ"])
df_raw["egfr_value"] = df_raw["egfr_value"].where(
    df_raw["egfr_value"].between(1, 200)
)

df_raw["ckd_g3plus_proxy"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="Int64"
)
has_egfr = df_raw["egfr_value"].notna()
df_raw.loc[has_egfr, "ckd_g3plus_proxy"] = (
    df_raw.loc[has_egfr, "egfr_value"] < 60
).astype(int)

df_raw["creatinine_value"] = numeric_series(df_raw["креатинин "])
df_raw["total_cholesterol"] = numeric_series(df_raw["холестерин"])
df_raw["ldl_mmol_l"] = numeric_series(df_raw["ЛПНП, ммоль/л"])
df_raw["crp_value"] = numeric_series(df_raw["СРБ"])
df_raw["glucose_value"] = numeric_series(df_raw["Глюкоза"])
df_raw["glucose_elevated_proxy"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="Int64"
)
has_glucose = df_raw["glucose_value"].notna()
df_raw.loc[has_glucose, "glucose_elevated_proxy"] = (
    df_raw.loc[has_glucose, "glucose_value"]
    >= GLUCOSE_ELEVATED_THRESHOLD_MMOL_L
).astype(int)
df_raw["bmi"] = numeric_series(df_raw["ИМТ"])
df_raw["bmi"] = df_raw["bmi"].where(df_raw["bmi"].between(10, 80))

df_raw["calcification_recorded"] = combine_binary_strict(
    documented_calcification(df_raw["кальциноз КА"]),
    documented_calcification(df_raw["кальциниты"]),
    documented_calcification(df_raw["КТ БЦА кальцинаты"]),
    documented_calcification(df_raw["кальциноз"])
)

df_raw["hospital_death"] = (
    1 - pd.to_numeric(
        df_raw["Исход 1- жив,0 -мерт"],
        errors="raise"
    )
).astype("Int64")

column_names = pd.Series(df_raw.columns, dtype="string")

named_columns = (
    column_names.notna()
    & column_names.str.strip().ne("")
    & ~column_names.str.match(r"^Unnamed", case=False, na=False)
)

df_raw = df_raw.loc[:, named_columns.to_numpy()]

empty_columns = df_raw.apply(
    lambda column: (
        column.isna()
        | column.astype("string").str.strip().eq("")
    ).all()
)

df_raw = df_raw.loc[:, ~empty_columns].copy()

print(
    "Подготовлено:",
    len(df_raw),
    "госпитализаций ×",
    df_raw.shape[1],
    "переменных"
)
print(
    df_raw["hospital_death"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Жив", 1: "Умер"})
    .to_string()
)

In [ ]:
df_raw.to_csv('df_raw.csv', index=False, encoding='utf-8-sig')

In [ ]:
def parse_operation_dates(series):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.replace({"-": pd.NA, "": pd.NA})
    cleaned = cleaned.str.replace(
        r"^0?2(\d{2})(-\d{2}-\d{2})",
        r"20\1\2",
        regex=True
    )

    return pd.to_datetime(
        cleaned,
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )


def parse_birth_dates(series):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.replace({"-": pd.NA, "": pd.NA})

    birth_date = pd.Series(
        pd.NaT,
        index=series.index,
        dtype="datetime64[ns]"
    )

    slash_format = cleaned.str.contains("/", na=False)
    dot_format = cleaned.str.contains(r"\.", regex=True, na=False)
    other_format = ~(slash_format | dot_format) & cleaned.notna()

    birth_date.loc[slash_format] = pd.to_datetime(
        cleaned.loc[slash_format],
        errors="coerce",
        dayfirst=False,
        format="mixed"
    )

    birth_date.loc[dot_format] = pd.to_datetime(
        cleaned.loc[dot_format],
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )

    birth_date.loc[other_format] = pd.to_datetime(
        cleaned.loc[other_format],
        errors="coerce",
        dayfirst=True,
        format="mixed"
    )

    return birth_date


def numeric_value(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (pd.Timestamp, datetime, date)):
        return float(to_excel(value))
    text = str(value).strip().replace("\u00a0", " ").replace(",", ".")
    match = re.search(r"[-+]?\d+(?:\.\d+)?", text)
    return float(match.group()) if match else np.nan


def numeric_series(series):
    return series.map(numeric_value).astype(float)


def coded_binary(series, blank_means_zero=False):
    normalized = series.astype("string").str.strip().str.lower()
    mapping = {
        "+": 1,
        "1": 1,
        "1.0": 1,
        "да": 1,
        "yes": 1,
        "true": 1,
        "-": 0,
        "0": 0,
        "0.0": 0,
        "нет": 0,
        "no": 0,
        "false": 0,
    }
    result = normalized.map(mapping)
    if blank_means_zero:
        result = result.fillna(0)
    return result.astype("Int64")


def combine_binary_strict(*series_list):
    values = pd.concat(series_list, axis=1)
    result = pd.Series(
        pd.NA,
        index=values.index,
        dtype="Int64"
    )
    any_positive = values.eq(1).any(axis=1)
    all_negative = values.notna().all(axis=1) & values.eq(0).all(axis=1)
    result.loc[any_positive] = 1
    result.loc[all_negative] = 0
    return result


def infer_sex(value):
    if pd.isna(value):
        return pd.NA
    tokens = re.sub(r"[^а-яё -]", " ", str(value).lower()).split()
    if any(token.endswith(("вна", "ична", "инична")) for token in tokens):
        return "female"
    if any(token.endswith(("вич", "ьич")) for token in tokens):
        return "male"
    if tokens and tokens[0].endswith(("ова", "ева", "ёва", "ина", "ына", "ая", "ская", "цкая")):
        return "female"
    if tokens and tokens[0].endswith(("ов", "ев", "ёв", "ин", "ын", "ий", "ый", "ской", "цкий")):
        return "male"
    return pd.NA


def documented_calcification(series):
    normalized = series.astype("string").str.strip().str.lower()
    result = pd.Series(
        pd.NA,
        index=series.index,
        dtype="Int64"
    )
    marker_positive = normalized.isin(["+", "1", "1.0", "да", "yes", "true"])
    marker_negative = normalized.isin(["-", "0", "0.0", "нет", "no", "false"])
    negative_text = normalized.str.contains(
        r"без[^.;]*кальцин|кальцин\w*[^.;]*не выяв",
        regex=True,
        na=False
    )
    positive_text = normalized.str.contains(
        r"кальцин|обызвеств|кальцинир",
        regex=True,
        na=False
    ) & ~negative_text
    result.loc[marker_negative | negative_text] = 0
    result.loc[marker_positive | positive_text] = 1
    return result


def boundary_index(frame, name_fragment):
    names = frame["ФИО"].astype("string").str.strip()
    matches = frame.index[
        names.str.contains(
            re.escape(name_fragment),
            case=False,
            regex=True,
            na=False
        )
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Граница {name_fragment}: найдено совпадений {len(matches)}"
        )
    return int(matches[0])


operation_date = parse_operation_dates(df_raw["Дата операции"])
birth_date = parse_birth_dates(df_raw["Год рождения"])

future_birth_date = (
    operation_date.notna()
    & birth_date.notna()
    & (birth_date > operation_date)
)

birth_date.loc[future_birth_date] = (
    birth_date.loc[future_birth_date]
    - pd.DateOffset(years=100)
)

valid_dates = operation_date.notna() & birth_date.notna()

birthday_not_reached = (
    (operation_date.dt.month < birth_date.dt.month)
    | (
        (operation_date.dt.month == birth_date.dt.month)
        & (operation_date.dt.day < birth_date.dt.day)
    )
)

df_raw["age_years"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="Int64"
)

df_raw.loc[valid_dates, "age_years"] = (
    operation_date.loc[valid_dates].dt.year
    - birth_date.loc[valid_dates].dt.year
    - birthday_not_reached.loc[valid_dates].astype(int)
).astype("Int64")

df_raw["age_years"] = df_raw["age_years"].where(
    df_raw["age_years"].between(18, 105)
)

df_raw = (
    df_raw
    .dropna(subset=["age_years"])
    .reset_index(drop=True)
)

df_raw["sex"] = df_raw["ФИО"].map(infer_sex).astype("string")
manual_male = (
    df_raw["ФИО"]
    .astype("string")
    .str.contains(
        "ANONYMIZED_PERSON",
        case=False,
        na=False
    )
)
df_raw.loc[manual_male, "sex"] = "male"
df_raw["female"] = (
    df_raw["sex"]
    .map({"female": 1, "male": 0})
    .astype("Int64")
)

df_raw["ef_percent"] = numeric_series(df_raw["Узи сердца ФВ %"])
df_raw["ef_percent"] = df_raw["ef_percent"].where(
    df_raw["ef_percent"] > 1.5,
    df_raw["ef_percent"] * 100
)
df_raw["ef_percent"] = df_raw["ef_percent"].where(
    df_raw["ef_percent"].between(10, 90)
)

df_raw["ef_category"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="string"
)
df_raw.loc[df_raw["ef_percent"] < 40, "ef_category"] = "<40"
df_raw.loc[
    df_raw["ef_percent"].between(40, 50, inclusive="both"),
    "ef_category"
] = "40–50"
df_raw.loc[df_raw["ef_percent"] > 50, "ef_category"] = ">50"

df_raw["stroke_history"] = coded_binary(
    df_raw["ОНМК анамнез (да/нет)"]
)
df_raw["diabetes"] = coded_binary(
    df_raw["диабет"]
)
df_raw["mi_acute_at_index"] = coded_binary(
    df_raw["Инфаркт миокарда острый"]
)
df_raw["mi_history"] = coded_binary(
    df_raw["Инфаркт анамнез (да/нет)"]
)
df_raw["ihd_recorded"] = coded_binary(
    df_raw["ИБС"]
)
df_raw["pci_history"] = coded_binary(
    df_raw["ЧТКА"]
)

af_paroxysmal = coded_binary(
    df_raw["ФП пароксизмальная"]
)
af_other = coded_binary(
    df_raw["ФП"]
)

df_raw["atrial_fibrillation"] = combine_binary_strict(
    af_paroxysmal,
    af_other
)

df_raw["prior_vascular_surgery"] = coded_binary(
    df_raw["любая операция на сосудах до индексного события"]
)

df_raw["ihd_before_index"] = combine_binary_strict(
    df_raw["ihd_recorded"],
    df_raw["mi_history"],
    df_raw["pci_history"]
)

df_raw["egfr_value"] = numeric_series(df_raw["СКФ"])
df_raw["egfr_value"] = df_raw["egfr_value"].where(
    df_raw["egfr_value"].between(1, 200)
)

df_raw["ckd_g3plus_proxy"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="Int64"
)
has_egfr = df_raw["egfr_value"].notna()
df_raw.loc[has_egfr, "ckd_g3plus_proxy"] = (
    df_raw.loc[has_egfr, "egfr_value"] < 60
).astype(int)

df_raw["creatinine_value"] = numeric_series(df_raw["креатинин "])
df_raw["total_cholesterol"] = numeric_series(df_raw["холестерин"])
df_raw["ldl_mmol_l"] = numeric_series(df_raw["ЛПНП, ммоль/л"])
df_raw["crp_value"] = numeric_series(df_raw["СРБ"])
df_raw["glucose_value"] = numeric_series(df_raw["Глюкоза"])
df_raw["glucose_elevated_proxy"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="Int64"
)
has_glucose = df_raw["glucose_value"].notna()
df_raw.loc[has_glucose, "glucose_elevated_proxy"] = (
    df_raw.loc[has_glucose, "glucose_value"]
    >= GLUCOSE_ELEVATED_THRESHOLD_MMOL_L
).astype(int)
df_raw["bmi"] = numeric_series(df_raw["ИМТ"])
df_raw["bmi"] = df_raw["bmi"].where(df_raw["bmi"].between(10, 80))

df_raw["calcification_recorded"] = combine_binary_strict(
    documented_calcification(df_raw["кальциноз КА"]),
    documented_calcification(df_raw["кальциниты"]),
    documented_calcification(df_raw["КТ БЦА кальцинаты"]),
    documented_calcification(df_raw["кальциноз"])
)

df_raw["hospital_death"] = (
    1 - pd.to_numeric(
        df_raw["Исход 1- жив,0 -мерт"],
        errors="raise"
    )
).astype("Int64")

peripheral_start = boundary_index(
    df_raw,
    "ANONYMIZED_PERSON"
)
peripheral_end = boundary_index(
    df_raw,
    "ANONYMIZED_PERSON"
)
absh_start = boundary_index(
    df_raw,
    "ANONYMIZED_PERSON"
)
absh_end = boundary_index(
    df_raw,
    "ANONYMIZED_PERSON"
)
carotid_start = boundary_index(
    df_raw,
    "ANONYMIZED_PERSON"
)
carotid_end = boundary_index(
    df_raw,
    "ANONYMIZED_PERSON"
)

if not (
    peripheral_start <= peripheral_end
    < absh_start <= absh_end
    < carotid_start <= carotid_end
):
    raise ValueError("Нарушен порядок границ клинических групп")

df_raw["vascular_group"] = pd.Series(
    pd.NA,
    index=df_raw.index,
    dtype="string"
)

df_raw.loc[
    peripheral_start:peripheral_end,
    "vascular_group"
] = "peripheral"

df_raw.loc[
    absh_start:absh_end,
    "vascular_group"
] = "absh_aortic_aneurysm"

df_raw.loc[
    carotid_start:carotid_end,
    "vascular_group"
] = "carotid"

df_raw = (
    df_raw
    .dropna(subset=["vascular_group"])
    .reset_index(drop=True)
)

rare_complete_case_columns = [
    "female",
    "mi_history",
    "stroke_history",
    "ldl_mmol_l",
]

df_raw = (
    df_raw
    .dropna(subset=rare_complete_case_columns)
    .reset_index(drop=True)
)

analysis_columns = [
    "vascular_group",
    "hospital_death",
    "age_years",
    "female",
    "prior_vascular_surgery",
    "ef_percent",
    "ef_category",
    "mi_history",
    "stroke_history",
    "ckd_g3plus_proxy",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "glucose_elevated_proxy",
    "crp_value",
    "egfr_value",
    "bmi",
]

df_raw = df_raw[analysis_columns].copy()

integer_columns = [
    "hospital_death",
    "age_years",
    "female",
    "prior_vascular_surgery",
    "mi_history",
    "stroke_history",
    "ckd_g3plus_proxy",
    "glucose_elevated_proxy",
]

for column in integer_columns:
    df_raw[column] = pd.to_numeric(
        df_raw[column],
        errors="coerce"
    ).astype("Int64")

float_columns = [
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

for column in float_columns:
    df_raw[column] = pd.to_numeric(
        df_raw[column],
        errors="coerce"
    ).astype(float)

processed = df_raw.copy()

df_peripheral = (
    processed.loc[
        processed["vascular_group"] == "peripheral"
    ]
    .reset_index(drop=True)
)

df_absh_aorta = (
    processed.loc[
        processed["vascular_group"] == "absh_aortic_aneurysm"
    ]
    .reset_index(drop=True)
)

df_carotid = (
    processed.loc[
        processed["vascular_group"] == "carotid"
    ]
    .reset_index(drop=True)
)


group_summary = (
    processed
    .groupby("vascular_group", observed=True)
    .agg(
        patients=("hospital_death", "size"),
        deaths=("hospital_death", "sum")
    )
    .reset_index()
)

print(
    "Подготовлено:",
    len(processed),
    "госпитализаций ×",
    processed.shape[1],
    "переменных"
)
display(group_summary)


## 3. Сохранение разделенных данных

In [ ]:
display(df_raw)

In [ ]:
processed.to_csv(
    OUTPUT_DIR / "hospital_outcomes_prepared.csv",
    index=False,
    encoding="utf-8-sig"
)
df_peripheral.to_csv(
    OUTPUT_DIR / "hospital_outcomes_peripheral.csv",
    index=False,
    encoding="utf-8-sig"
)
df_absh_aorta.to_csv(
    OUTPUT_DIR / "hospital_outcomes_absh_aortic_aneurysm.csv",
    index=False,
    encoding="utf-8-sig"
)
df_carotid.to_csv(
    OUTPUT_DIR / "hospital_outcomes_carotid.csv",
    index=False,
    encoding="utf-8-sig"
)

## 4. Контроль качества и пропусков

`glucose_value` — исходный уровень глюкозы, ммоль/л. `glucose_elevated_proxy` — бинарный аналитический признак: 1 при глюкозе ≥5,6 ммоль/л, 0 при значении <5,6 ммоль/л; при отсутствии измерения сохраняется пропуск. Поскольку статус натощак не указан, признак является прокси, а не диагнозом.


In [ ]:
continuous_features = [
    "age_years",
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

binary_features = [
    "female",
    "prior_vascular_surgery",
    "mi_history",
    "stroke_history",
    "ckd_g3plus_proxy",
    "glucose_elevated_proxy",
]

primary_features = (
    continuous_features
    + binary_features
    + ["ef_category"]
)

label_map = {
    "age_years": "Возраст, полных лет",
    "ef_percent": "Фракция выброса, %",
    "creatinine_value": "Креатинин",
    "total_cholesterol": "Общий холестерин",
    "ldl_mmol_l": "ЛПНП, ммоль/л",
    "glucose_value": "Глюкоза, ммоль/л",
    "crp_value": "С-реактивный белок",
    "egfr_value": "СКФ",
    "bmi": "Индекс массы тела",
    "female": "Женский пол",
    "prior_vascular_surgery": "Операция на сосудах до индексного события",
    "mi_history": "Инфаркт миокарда в анамнезе",
    "stroke_history": "Инсульт в анамнезе",
    "ckd_g3plus_proxy": "ХБП стадии G3 и выше, прокси",
    "glucose_elevated_proxy": "Повышенная глюкоза ≥5,6 ммоль/л, прокси",
    "ef_category": "Категория фракции выброса",
}

missingness = pd.DataFrame(
    {
        "variable": primary_features,
        "label_ru": [
            label_map.get(variable, variable)
            for variable in primary_features
        ],
        "available_n": [
            int(processed[variable].notna().sum())
            for variable in primary_features
        ],
        "missing_n": [
            int(processed[variable].isna().sum())
            for variable in primary_features
        ],
        "missing_pct": [
            round(processed[variable].isna().mean() * 100, 1)
            for variable in primary_features
        ],
    }
).sort_values(
    "missing_pct",
    ascending=False
).reset_index(drop=True)
display(missingness)

## 5. Проверка числовых показателей

Диапазоны используются как технические границы для поиска подозрительных записей. Они не заменяют проверку единиц измерения и исходной медицинской документации.

In [ ]:
numeric_audit_features = [
    "age_years",
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

numeric_audit = (
    processed[numeric_audit_features]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    )
    .T
)

numeric_audit.insert(
    1,
    "missing_n",
    processed[numeric_audit_features].isna().sum()
)
numeric_audit.insert(
    2,
    "missing_pct",
    (
        processed[numeric_audit_features]
        .isna()
        .mean()
        .mul(100)
        .round(1)
    )
)

display(numeric_audit)


In [ ]:
quality_limits = {
    "age_years": (18, 105),
    "ef_percent": (10, 90),
    "creatinine_value": (20, 1500),
    "total_cholesterol": (1, 15),
    "ldl_mmol_l": (0.1, 15),
    "glucose_value": (2, 40),
    "crp_value": (0, 500),
    "egfr_value": (1, 200),
    "bmi": (10, 80),
}

range_audit = pd.DataFrame(
    [
        {
            "variable": variable,
            "lower_limit": lower_limit,
            "upper_limit": upper_limit,
            "below_n": int((processed[variable] < lower_limit).sum()),
            "above_n": int((processed[variable] > upper_limit).sum()),
        }
        for variable, (lower_limit, upper_limit)
        in quality_limits.items()
    ]
)

range_flag_rows = []

for variable, (lower_limit, upper_limit) in quality_limits.items():
    flagged = processed[variable].notna() & (
        (processed[variable] < lower_limit)
        | (processed[variable] > upper_limit)
    )
    if flagged.any():
        current_flags = processed.loc[
            flagged,
            ["vascular_group", "hospital_death", variable]
        ].copy()
        current_flags.insert(0, "row_index", current_flags.index)
        current_flags.insert(1, "variable", variable)
        current_flags = current_flags.rename(
            columns={variable: "value"}
        )
        range_flag_rows.append(current_flags)

range_flag_rows = (
    pd.concat(range_flag_rows, ignore_index=True)
    if range_flag_rows
    else pd.DataFrame(
        columns=[
            "row_index",
            "variable",
            "vascular_group",
            "hospital_death",
            "value",
        ]
    )
)

display(range_audit)
display(range_flag_rows)


In [ ]:
numeric_missing_by_group = (
    processed
    .groupby("vascular_group", observed=True)[numeric_audit_features]
    .agg(lambda series: round(series.isna().mean() * 100, 1))
)

numeric_missing_by_outcome = (
    processed
    .groupby("hospital_death", observed=True)[numeric_audit_features]
    .agg(lambda series: round(series.isna().mean() * 100, 1))
    .rename(index={0: "Жив", 1: "Умер"})
)

display(numeric_missing_by_group)
display(numeric_missing_by_outcome)


## 6. Дополнительный датасет с импутацией

Дополнительный набор предназначен для исследовательского анализа. Пропуски заполняются медианой внутри сочетания сосудистой группы и пола, затем медианой сосудистой группы и общей медианой. ФВ и прокси ХБП после заполнения рассчитываются повторно.

In [ ]:
imputation_targets = [
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

processed_imputed = processed.copy()

for variable in imputation_targets:
    missing_indicator = f"{variable}_was_missing"
    processed_imputed[missing_indicator] = (
        processed_imputed[variable]
        .isna()
        .astype("Int64")
    )

    group_sex_median = (
        processed_imputed
        .groupby(
            ["vascular_group", "female"],
            observed=True
        )[variable]
        .transform("median")
    )

    group_median = (
        processed_imputed
        .groupby("vascular_group", observed=True)[variable]
        .transform("median")
    )

    processed_imputed[variable] = (
        processed_imputed[variable]
        .fillna(group_sex_median)
        .fillna(group_median)
        .fillna(processed_imputed[variable].median())
    )


In [ ]:
processed_imputed["ef_category"] = pd.Series(
    pd.NA,
    index=processed_imputed.index,
    dtype="string"
)
processed_imputed.loc[
    processed_imputed["ef_percent"] < 40,
    "ef_category"
] = "<40"
processed_imputed.loc[
    processed_imputed["ef_percent"].between(
        40,
        50,
        inclusive="both"
    ),
    "ef_category"
] = "40–50"
processed_imputed.loc[
    processed_imputed["ef_percent"] > 50,
    "ef_category"
] = ">50"

processed_imputed["ckd_g3plus_proxy"] = (
    processed_imputed["egfr_value"] < 60
).astype("Int64")

processed_imputed["glucose_elevated_proxy"] = (
    processed_imputed["glucose_value"]
    >= GLUCOSE_ELEVATED_THRESHOLD_MMOL_L
).astype("Int64")

remaining_imputation_missing = (
    processed_imputed[imputation_targets]
    .isna()
    .sum()
)

print(
    "Дополнительный датасет:",
    len(processed_imputed),
    "строк ×",
    processed_imputed.shape[1],
    "столбцов"
)
display(remaining_imputation_missing.to_frame("missing_n"))


In [ ]:
imputation_summary = pd.DataFrame(
    [
        {
            "variable": variable,
            "imputed_n": int(processed[variable].isna().sum()),
            "missing_pct_before": round(
                processed[variable].isna().mean() * 100,
                1
            ),
            "observed_median": processed[variable].median(),
            "filled_values_median": processed_imputed.loc[
                processed[variable].isna(),
                variable
            ].median(),
            "median_after": processed_imputed[variable].median(),
            "q25_after": processed_imputed[variable].quantile(0.25),
            "q75_after": processed_imputed[variable].quantile(0.75),
        }
        for variable in imputation_targets
    ]
)

display(imputation_summary)


In [ ]:
imputed_medians_by_outcome = (
    processed_imputed
    .groupby("hospital_death", observed=True)[imputation_targets]
    .median()
    .rename(index={0: "Жив", 1: "Умер"})
    .T
)

imputed_medians_by_group = (
    processed_imputed
    .groupby("vascular_group", observed=True)[imputation_targets]
    .median()
    .T
)

display(imputed_medians_by_outcome)
display(imputed_medians_by_group)


In [ ]:
processed_imputed.to_csv(
    OUTPUT_DIR / "hospital_outcomes_imputed_exploratory.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Если ваша итоговая таблица называется df_raw, используйте её:
df_plot = df_raw.copy() 

# Список переменных в том порядке, как на вашем скриншоте
variables = [
    'age_years',        # Возраст
    'creatinine_value', # Креатинин
    'crp_value',        # С-реактивный белок
    'bmi',              # ИМТ
    'glucose_value'     # Глюкоза
]

# Задаем цвета и для числовых, и для строковых значений
palette = {
    0: '#466c8b', 1: '#be7488', 
    '0': '#466c8b', '1': '#be7488'
}

# Создаем сетку графиков: 2 строки, 3 колонки
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 8))
axes = axes.flatten() # Вытягиваем в плоский массив для удобного цикла

# Проходимся циклом по переменным и рисуем графики
for i, var in enumerate(variables):
    ax = axes[i]
    
    # Пропускаем, если переменной почему-то нет в таблице
    if var not in df_plot.columns:
        continue
        
    # Строим boxplot
    sns.boxplot(
        data=df_plot,
        x='hospital_death',
        y=var,
        hue='hospital_death', 
        order=[0, 1],     # Фиксируем порядок: 0 (Живы) слева, 1 (Умерли) справа
        palette=palette,
        showfliers=False, # Скрываем выбросы-точки
        width=0.6,        # Ширина ящиков
        linewidth=1.2,    # Толщина линий
        legend=False,     # Прячем легенду
        ax=ax
    )
    
    # 1. Подсчет реального числа пациентов (n) без учета пропусков
    n_alive = df_plot.loc[df_plot['hospital_death'] == 0, var].count()
    n_dead = df_plot.loc[df_plot['hospital_death'] == 1, var].count()
    
    # 2. УБИРАЕМ ВСЕ СЛОВА с осей Y и заголовков
    ax.set_title('')       
    ax.set_xlabel('')      
    ax.set_ylabel('')      
    
    # 3. Добавляем подписи n=... на ось X
    ax.set_xticks([0, 1])
    # Если вы хотите убрать слова "Живы/Умерли" и оставить СТРОГО только "n=...",
    # замените строку ниже на: ax.set_xticklabels([f'n={n_alive}', f'n={n_dead}'])
    ax.set_xticklabels([f'n = {n_alive}', f'n = {n_dead}'])
    
    # Визуальный тюнинг (сетка и рамки)
    sns.despine(ax=ax)
    ax.yaxis.grid(True, color='lightgray', linestyle='-', alpha=0.7)
    ax.set_axisbelow(True)

# Удаляем пустой 6-й график (нижний правый угол)
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig('overall_sample_boxplots.png', dpi=300, bbox_inches='tight')

# Сохраняем в векторном формате EPS
fig.savefig('overall_sample_boxplots.eps', format='eps', bbox_inches='tight')
# ===========================